# Cắt keyframe — phần 3/5

Notebook này cắt **6 batch**: `K13` · `K14` · `K15` · `K16` · `K17` · `K18` — tổng **181 video**.
Mở lên, chạy hết các ô từ trên xuống. **Không phải sửa gì.**

| | |
|---|---|
| **Đầu vào** | `/content/drive/MyDrive/aic25/AIC_Video` |
| **Batch của bạn** | `K13` (30) · `K14` (30) · `K15` (31) · `K16` (32) · `K17` (30) · `K18` (28) |
| **Tổng** | **181 video** |
| **Đầu ra** | `/content/drive/MyDrive/aic26/output_demo/3/v001/` |
| **Model** | TransNetV2 |

## Cả nhóm chia thế nào

| Phần | Batch | Số video |
|---|---|---:|
| **1** | `K01 `·` K02 `·` K03 `·` K04 `·` K05 `·` K06` | 183 |
| **2** | `K07 `·` K08 `·` K09 `·` K10 `·` K11 `·` K12` | 179 |
| **3** | `K13 `·` K14 `·` K15 `·` K16 `·` K17 `·` K18` | 181 |
| **4** | `K19 `·` K20 `·` L21 `·` L22 `·` L23 `·` L24` | 190 |
| **5** | `L25 `·` L26 `·` L27 `·` L28 `·` L29 `·` L30` | 745 |

Chia theo batch liền nhau nên **không đều**: `L26` một mình có 498 video, khiến phần 5
nặng gấp 4 lần. Đây là lựa chọn có ý thức — đổi lấy việc mỗi người nắm trọn vài batch,
dễ nói chuyện và dễ kiểm hơn là mỗi người một mớ video rải rác.

## Chuẩn bị một lần cho cả nhóm

Chép `data_raw/fps_map.json` từ repo lên `/content/drive/MyDrive/aic26/fps_map.json`.

BTC dùng chính bảng đó để đánh số frame trong đáp án. Không có nó thì mọi video phải dò
fps bằng `ffprobe`, mà `ffprobe` có thể trả `30.0` ở video BTC ghi `29.97` — lệch 20%,
tới phút 20 là sai gần 4 phút, đủ trượt hẳn cảnh chứa đáp án.

> ✓ **Bảng đã đủ và đã kiểm.** 1478 video (873 của L21–L30 + 605 của K01–K20). Đối chiếu
> với `ffprobe` trên từng file: **1478/1478 khớp**, lệch lớn nhất 0,002 fps trên một video
> 35 giây. Notebook sẽ báo `✓ Mọi video đều có trong bảng` — nếu thấy con số khác 0 ở dòng
> "phải dò", nghĩa là bảng trên Drive đã cũ, chép lại từ repo.

> ⚠️ **Đừng làm tròn lại `29.97`.** Giá trị toán học chính xác là `30000/1001 = 29.970030`,
> nhưng dùng nó sẽ lệch 1 frame ở **54%** số dòng của 30 video đó. `29.97` như bảng ghi mới
> tái lập đúng `frame_idx` của BTC — kiểm trên 13.852 dòng, khớp 100%.

## Cách cắt

```
TransNetV2 → cảnh → lấy mẫu THEO TỐC ĐỘ → lưu JPEG độ phân giải GỐC
```

| | |
|---|---|
| **Số ảnh mỗi cảnh** | `n = 2` tới 1,67 giây, sau đó cứ 2 giây thêm 1 ảnh, kẹp `[2, 40]` |
| **Lùi biên** | 3 frame mỗi đầu, tránh frame chuyển cảnh |
| **Lọc trùng** | **không** — paper §3.2.3 lọc sau khi trích đặc trưng |
| **Độ phân giải** | giữ nguyên gốc — thu nhỏ về 960 làm OCR tụt 44% → 34% |

@30fps: `8–50 frame → 2 ảnh · 51–110 → 3 · 111–170 → 4`
@25fps: `8–42 frame → 2 ảnh · 43–92 → 3 · 93–142 → 4`

## Colab ngắt giữa chừng thì sao

Mở lại, chạy lại từ đầu. `progress.json` ghi nhớ video nào xong rồi, `SECTION 02b` đưa
những video đã cắt mà chưa kịp nén trở lại hàng đợi. Không mất gì, không làm lại.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 02a – Hằng số của notebook này
# ═══════════════════════════════════════════════════════════════════════════════
# %%
# Notebook 3/5. Đã đặt sẵn, KHÔNG phải sửa gì để chạy.

PART    = 3
VERSION = "v001"   # tăng lên "v002" khi đổi CÁCH LÀM, không phải khi chạy thêm video

# Sáu batch của phần này. Notebook chỉ cắt video thuộc các batch dưới đây.
BATCHES = ["K13", "K14", "K15", "K16", "K17", "K18"]

# Số video kỳ vọng, đếm từ bảng fps của BTC:
#   30 + 30 + 31 + 32 + 30 + 28 = 181
EXPECTED_VIDEOS = 181

# CẢ 5 PHẦN GHI CHUNG MỘT THƯ MỤC — mọi zip nằm cùng chỗ, bước dựng index
# chỉ cần trỏ vào đây là thấy đủ, không phải gộp tay từ 5 nơi.
#
# Để chung được, hai thứ đã được tách theo phần ở SECTION 02 và 06:
#   progress_p{PART}.json          tiến độ riêng, không đè nhau
#   AIC2026_frames_p{PART}_NNN.zip  tên zip riêng, không đè nhau
DRIVE_OUT  = f"/content/drive/MyDrive/aic26/Frame_Transnet_2026_{VERSION}"

# Dùng chung cho cả 5 người — trọng số TransNetV2 và bảng fps chỉ tải một lần.
SHARED_DIR = "/content/drive/MyDrive/aic26"

print(f"[Part]     {PART}/5")
print(f"[Batch]    {' '.join(BATCHES)}")
print(f"[Expected] {EXPECTED_VIDEOS} videos")
print(f"[Version]  {VERSION}")
print(f"[Output]   {DRIVE_OUT}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 00 – Install Dependencies
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import subprocess, sys

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

print("[Install] Installing baseline dependencies...")
_pip("--upgrade", "transnetv2-pytorch==1.0.5", "ffmpeg-python")
_pip("open_clip_torch")
_pip("faiss-cpu")   # faiss-gpu trên PyPI dừng ở 2022, không cài được
_pip("tqdm")
_pip("transformers>=4.40.0", "Pillow", "opencv-python-headless")
_pip("gdown")
# BEiT3 (unilm) — dependency thật, verify bằng cách tự cài + test trong sandbox
# trước khi đưa vào đây. deepspeed KHÔNG cần cho inference (verify: chỉ dùng
# trong run_beit3_finetuning.py, không import trong modeling_finetune.py).
_pip("torchscale==0.2.0", "timm==0.4.12", "torchmetrics==0.7.3")
_pip("tensorboardX", "ftfy", "blobfile", "sentencepiece")
# setuptools<81: torchmetrics 0.7.3 dùng pkg_resources (deprecated, bị xóa
# khỏi setuptools>=81) — verify lỗi thật này bằng cách tự chạy import và bắt
# ModuleNotFoundError trong sandbox trước khi thêm dòng pin này.
_pip("setuptools<81")
print("[Install] All done\n")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 01 – Mount Drive  ← PHẢI mount TRƯỚC mọi thứ liên quan /content/drive
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import os
DRIVE_MOUNT = "/content/drive"
if os.path.ismount(DRIVE_MOUNT):
    print("[Drive] Already mounted.")
else:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
    print("[Drive] Mounted OK.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 02 – Config & Dirs  ← SAU khi Drive đã mount
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import sys, json, glob, time, math, zipfile, shutil, subprocess as _sp
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import defaultdict

# ─── MODE ────────────────────────────────────────────────────────────────────
# "TEST"  → chỉ xử lý TEST_VIDEOS (~83 videos, đủ để kiểm thử pipeline)
# "FULL"  → toàn bộ ~1478 videos (production run)
MODE = "TEST"   # ← đổi thành "FULL" khi cần chạy thật

# ─── PATHS (path cứng, không đổi) ────────────────────────────────────────────
VIDEO_ROOT   = "/content/drive/MyDrive/aic25/AIC_Video"
OUTPUT_ROOT  = DRIVE_OUT
CKPT_TNV2    = f"{SHARED_DIR}/transnetv2-pytorch-weights.pth"
LOCAL_KF_DIR = "/content/kf_buffer"

# ─── TEST VIDEOS ──────────────────────────────────────────────────────────────
# Đối chiếu lại với All_Queries.docx (bản đầy đủ, verify từng câu qua raw XML,
# không dựa vào comment cũ). Round 1 (25 câu) + Round 2 (30 câu) đã full fill,
# Round 3 CHƯA có nội dung trong docx này (chỉ có tiêu đề, không có query nào).
#
# Round 1 GT videos — verify từng câu, KHỚP 100% với docx (24/25 câu có video;
# R1-21 không có video xác nhận — thay vào đó dùng K05_V006 làm benchmark thay
# thế, đúng như docx ghi chú: "Vid gần nhất giống mô tả... K05_V006 (khác mỗi
# ở chỗ không phải "bọ" mà là "chim")... nếu muốn lấy benchmark cho câu này
# thì lấy vid đó làm truth value")
_GT_L = {
    "L21_V015",  # R1-1 KIS: 4 phi hành gia mặc áo đen / tàu vũ trụ tư nhân
    "L21_V027",  # R1-14 KIS: điêu khắc cát, khói hồng
    "L21_V029",  # R1-2 KIS: đàn hổ con quý hiếm miền Nam
    "L22_V022",  # R1-23 KIS: thị trấn ven biển / cá mập (phim Spielberg 1975)
    "L22_V030",  # R1-8 KIS: lễ hội ẩm thực Nhật, cô bé đeo bạch tuộc đỏ
    "L23_V007",  # R1-24 KIS: đua xe đạp, 3 tay đua đồng phục trắng-vàng-xanh
    "L23_V017",  # R1-25 KIS: flycam đua xe đạp, áo xanh dương trắng vượt lên
    "L24_V033",  # R1-16 TRAKE: múa lân vàng đen trắng / lân quay cột
    "L25_V058",  # R2-21 QA: bài tập vận dụng, khoảng cách AB 2 chữ số, đáp C
    "L26_V004",  # R1-20 KIS: panna cotta 3 ly / nho đỏ bạc hà
    "L26_V056",  # R1-6 KIS: gỏi cuốn chay, bánh tráng vàng tím (video đúng)
    "L26_V072",  # R1-18 TRAKE: cắt nấm → củ năng → đậu hũ → bếp
    "L26_V178",  # R1-22 QA: bánh ít trần, 200g thịt nạc xay
    "L26_V194",  # R1-4 TRAKE: bột → măng tây → dầu → chiên
    "L26_V200",  # R1-12 KIS: trang trí bánh rán, chocolate + chuối + dâu
    "L26_V385",  # R1-6 KIS: gỏi cuốn chay (video tìm được, dù L26_V056 đúng hơn)
    "L27_V010",  # R1-19 QA: đình thần Nguyễn Trung Trực Kiên Giang / 2 câu thơ
    "L27_V013",  # R1-9 KIS: thu hoạch dứa miền Tây, bà cụ + cô gái áo hồng + ghe xanh
    "L27_V014",  # R1-5 KIS: cho dê ăn, áo thun trắng + áo kẻ sọc tím
    "L29_V023",  # R1-7 KIS: chim đầu đen ánh xanh mắt đỏ, rừng Nam Bộ
    "L30_V014",  # R2-11 KIS: phụ nữ lớn tuổi đọc sách / bìa ảnh cụ lão
    "L30_V017",  # R1-10 KIS: 3 người chơi nhạc cụ kim loại tròn / kệ sách
    "L30_V040",  # R2-18 KIS: mèo trong phòng, tô kim loại úp, bảng tên "Nà Ní"
    "L30_V057",  # R1-11 KIS: mảnh bìa đổ bóng thành chân dung / mũ lưỡi trai đen
    "L30_V072",  # R1-15 QA: câu lạc bộ FANA trao quà Xã Giang Ly Khánh Hòa
    "L30_V092",  # R1-17 KIS: trao quà từ thiện bệnh viện, phông đỏ COVID-19
    "L30_V095",  # R1-13 KIS: vệ sinh máy ảnh, ống kính + khăn tím hồng + tăm bông
}

# Round 2 GT videos (K-series = dataset batch K). Verify 100% với docx —
# 27/30 câu có video xác nhận (R2-16, R2-19 explicit "chưa tìm được đáp án"
# trong docx → KHÔNG có GT, cố tình bỏ qua. R2-12, R2-23 KHÔNG có video ID
# xác nhận trong docx dù trước đây từng nhớ nhầm là có → xem _LEGACY_UNCONFIRMED)
_GT_K = {
    "K01_V009",  # R2-4 KIS: cứu hộ lũ, cụ bà đội nón lá trên xuồng cam
    "K01_V018",  # R2-15 KIS: cô gái phỏng vấn + Deadpool + múa đao
    "K02_V005",  # R2-5 KIS + R2-22 TRAKE: robot vẽ tranh + sách MILK / khám xét
    "K03_V019",  # R2-1 KIS: nhiều người áo cờ đỏ sao vàng trước biểu tượng con cua
    "K03_V022",  # R2-29 KIS: cảng biển, cầu dài ra biển, người đội nón tai bèo
    "K03_V023",  # R2-8 KIS: cầu vòm thép đỏ → ca nô vệt sóng tròn
    "K05_V013",  # R2-24 KIS: ngư dân Campuchia nâng cá lớn nguy cấp
    "K05_V018",  # R2-25 KIS: robot 4 chân bánh xe trên đồng cỏ + bò
    "K05_V025",  # R2-7 KIS: người tuyết khổng lồ đội mũ + bình gốm băng tuyết
    "K06_V010",  # R2-20 KIS: cào muối đồng, đoàn người, "Gừng cay muối mặn"
    "K07_V025",  # R2-10 TRAKE: xưởng lắp ráp robot + công nhân + cửa ô tô
    "K07_V030",  # R2-17 KIS: phụ nữ đội nón len đeo kính "Happy New Year"
    "K09_V007",  # R2-30 KIS: kị sĩ cưỡi ngựa đồng cỏ xanh / cờ đỏ
    "K09_V012",  # R2-26 KIS: phỏng vấn Trump + đoàn người ven biển vác bao trắng
    "K09_V015",  # R2-13 KIS: bánh giá 9€ và 4.80€ / sản xuất 12 cái/tầng
    "K09_V018",  # R2-6 KIS: đài phun nước trăm ngọn nến / phụ nữ bên ly nến
    "K12_V001",  # R2-9 KIS: cá nhám 211kg treo trên xe cẩu
    "K14_V027",  # R2-2 KIS: lễ hội đèn lồng, hanbok, đèn lồng khổng lồ hình người
    "K15_V010",  # R2-27 KIS: họp giá điện, màn hình LED chữ trắng nền xanh
    "K16_V010",  # R2-14 KIS: icon hình não 5×10 / thống kê tác dụng phụ thuốc
    "K17_V003",  # R2-3 QA: bản đồ Việt Nam trái cây / Khu Du Lịch Quốc Gia Mộc Châu
    "K18_V002",  # R2-28 KIS: nhiều loài cá cảnh, cá đĩa, cá rồng, cá la hán
    "K05_V006",  # R1-21 BENCHMARK THAY THẾ (không phải GT gốc): docx ghi rõ
                 # câu gốc "năm ngoái BTC nói lỗi", K05_V006 là video gần nhất
                 # (khác "bọ" → "chim") được docx đề xuất dùng làm truth value
                 # nếu cần benchmark cho câu này.
}

# ⚠ LEGACY UNCONFIRMED — trước đây từng coi là GT nhưng All_Queries.docx (bản
# đầy đủ, verify qua raw XML) KHÔNG có video ID cho các câu này:
#   K04_V013 (từng gán cho R2-23) — R2-23 trong docx mới KHÔNG có dòng
#     "Kxx_Vxxx,timestamp" nào, chỉ có mô tả + nhận xét, không có đáp án.
#   K07_V019 (từng gán cho R2-12) — tương tự, R2-12 không có video ID.
#   L26_V176, L29_V020 (từng gán cho R3-1, R3-2) — docx này DỪNG NGAY tại
#     tiêu đề "Round 3", không có nội dung Round 3 nào cả, nên không thể
#     verify lại 2 video này từ nguồn hiện tại.
# Quyết định: KHÔNG xóa (đã cắt sẵn trong 95 video hiện có, xóa cũng không
# undo được), nhưng KHÔNG coi là GT đã verify — giữ trong diversity pool.
_LEGACY_UNCONFIRMED = {
    "K04_V013", "K07_V019", "L26_V176", "L29_V020",
}

# Diversity videos (NON-GT) — mở rộng để đạt ~25-30GB frame output (yêu cầu),
# thay vì chỉ 1-3 video/batch như trước (→ 8GB/95 video). Mở rộng lên
# K01-K18 × V001-V008 (144 video) + L21-L30 × V001-V010 (100 video) = 244 mới.
# Ước tính: 8GB/95 video ≈ 86MB frame-output/video trung bình → ~300 video
# tổng (GT+diversity) × 86MB ≈ 25.8GB, khớp mục tiêu 25-30GB. Video nào không
# tồn tại trên Drive sẽ tự động bị skip + WARN (đã có sẵn cơ chế ở Section 03).
_DIVERSITY = {
    "K01_V001", "K01_V002", "K01_V003", "K01_V004", "K01_V005", "K01_V006", "K01_V007", "K01_V008",
    "K02_V001", "K02_V002", "K02_V003", "K02_V004", "K02_V005", "K02_V006", "K02_V007", "K02_V008",
    "K03_V001", "K03_V002", "K03_V003", "K03_V004", "K03_V005", "K03_V006", "K03_V007", "K03_V008",
    "K04_V001", "K04_V002", "K04_V003", "K04_V004", "K04_V005", "K04_V006", "K04_V007", "K04_V008",
    "K05_V001", "K05_V002", "K05_V003", "K05_V004", "K05_V005", "K05_V006", "K05_V007", "K05_V008",
    "K06_V001", "K06_V002", "K06_V003", "K06_V004", "K06_V005", "K06_V006", "K06_V007", "K06_V008",
    "K07_V001", "K07_V002", "K07_V003", "K07_V004", "K07_V005", "K07_V006", "K07_V007", "K07_V008",
    "K08_V001", "K08_V002", "K08_V003", "K08_V004", "K08_V005", "K08_V006", "K08_V007", "K08_V008",
    "K09_V001", "K09_V002", "K09_V003", "K09_V004", "K09_V005", "K09_V006", "K09_V007", "K09_V008",
    "K10_V001", "K10_V002", "K10_V003", "K10_V004", "K10_V005", "K10_V006", "K10_V007", "K10_V008",
    "K11_V001", "K11_V002", "K11_V003", "K11_V004", "K11_V005", "K11_V006", "K11_V007", "K11_V008",
    "K12_V001", "K12_V002", "K12_V003", "K12_V004", "K12_V005", "K12_V006", "K12_V007", "K12_V008",
    "K13_V001", "K13_V002", "K13_V003", "K13_V004", "K13_V005", "K13_V006", "K13_V007", "K13_V008",
    "K14_V001", "K14_V002", "K14_V003", "K14_V004", "K14_V005", "K14_V006", "K14_V007", "K14_V008",
    "K15_V001", "K15_V002", "K15_V003", "K15_V004", "K15_V005", "K15_V006", "K15_V007", "K15_V008",
    "K16_V001", "K16_V002", "K16_V003", "K16_V004", "K16_V005", "K16_V006", "K16_V007", "K16_V008",
    "K17_V001", "K17_V002", "K17_V003", "K17_V004", "K17_V005", "K17_V006", "K17_V007", "K17_V008",
    "K18_V001", "K18_V002", "K18_V003", "K18_V004", "K18_V005", "K18_V006", "K18_V007", "K18_V008",
    "L21_V001", "L21_V002", "L21_V003", "L21_V004", "L21_V005", "L21_V006", "L21_V007", "L21_V008", "L21_V009", "L21_V010",
    "L22_V001", "L22_V002", "L22_V003", "L22_V004", "L22_V005", "L22_V006", "L22_V007", "L22_V008", "L22_V009", "L22_V010",
    "L23_V001", "L23_V002", "L23_V003", "L23_V004", "L23_V005", "L23_V006", "L23_V007", "L23_V008", "L23_V009", "L23_V010",
    "L24_V001", "L24_V002", "L24_V003", "L24_V004", "L24_V005", "L24_V006", "L24_V007", "L24_V008", "L24_V009", "L24_V010",
    "L25_V001", "L25_V002", "L25_V003", "L25_V004", "L25_V005", "L25_V006", "L25_V007", "L25_V008", "L25_V009", "L25_V010",
    "L26_V001", "L26_V002", "L26_V003", "L26_V004", "L26_V005", "L26_V006", "L26_V007", "L26_V008", "L26_V009", "L26_V010",
    "L27_V001", "L27_V002", "L27_V003", "L27_V004", "L27_V005", "L27_V006", "L27_V007", "L27_V008", "L27_V009", "L27_V010",
    "L28_V001", "L28_V002", "L28_V003", "L28_V004", "L28_V005", "L28_V006", "L28_V007", "L28_V008", "L28_V009", "L28_V010",
    "L29_V001", "L29_V002", "L29_V003", "L29_V004", "L29_V005", "L29_V006", "L29_V007", "L29_V008", "L29_V009", "L29_V010",
    "L30_V001", "L30_V002", "L30_V003", "L30_V004", "L30_V005", "L30_V006", "L30_V007", "L30_V008", "L30_V009", "L30_V010",
} | _LEGACY_UNCONFIRMED

# Hợp nhất: GT (~50 videos, verify 100% từ docx) + Diversity (~244+4 videos)
TEST_VIDEOS: set[str] = _GT_L | _GT_K | _DIVERSITY

print(f"[Mode] MODE = {MODE}")
print(f"[Mode] GT videos: {len(_GT_L)+len(_GT_K)} | Diversity: {len(_DIVERSITY)} | Total TEST set: {len(TEST_VIDEOS)}")

# ─── TRANSNETV2 ──────────────────────────────────────────────────────────────
TNV2_W        = 48
TNV2_H        = 27
CUT_THR       = 0.5
MIN_CUT_DIST  = 12
MIN_SCENE_LEN = 8
TNV2_CHUNK    = 200
TNV2_OVERLAP  = 25
TNV2_GDRIVE_ID = "1K7tpO6NMGVqCF30mwuyR-i2HZMA7Fsrg"

# ─── FRAME SELECTION (Paper Alg.1) ───────────────────────────────────────────
N_SAMPLE_PER_SCENE = 4
DEDUP_THR     = 0.90
JPEG_QUALITY  = 90

# ─── ZIP ─────────────────────────────────────────────────────────────────────
ZIP_EVERY_GB  = 2.0
ZIP_DIR       = f"{OUTPUT_ROOT}/zips"
INDEX_DIR     = f"{OUTPUT_ROOT}/indexes"
PROGRESS_FILE = f"{OUTPUT_ROOT}/progress_p{PART}.json"   # tách theo phần

# ─── MODELS ──────────────────────────────────────────────────────────────────
# BEiT3-large (Wang et al., CVPR 2023 — chính xác model [51] paper 2504.08384
# cite cho nhánh "fine-grained"). Checkpoint tốt nhất public cho retrieval:
# beit3_large_patch16_384_coco_retrieval.pth — 675M params, fine-tuned COCO
# retrieval, ITC dual-encoder. Verify TOÀN BỘ bằng cách tự clone source thật
# từ microsoft/unilm, cài dependency, patch lỗi torch._six, tải checkpoint
# thật (1.29GB) và chạy forward pass thành công (output 1024-dim, khớp
# encoder_embed_dim khai báo trong _get_large_config của chính unilm).
BEIT3_CKPT_URL   = "https://github.com/addf400/files/releases/download/beit3/beit3_large_patch16_384_coco_retrieval.pth"
BEIT3_SPM_URL    = "https://github.com/addf400/files/releases/download/beit3/beit3.spm"
BEIT3_MODEL_NAME = "beit3_large_patch16_384_retrieval"
BEIT3_INPUT_SIZE = 384
BEIT3_BATCH      = 8   # 675M params fp32 — conservative batch cho T4
BEIT3_CODE_DIR   = "/content/unilm_beit3"   # nơi clone source code thật
# ViT-bigG-14 (LAION, laion2b_s39b_b160k) — giữ nguyên theo quyết định trước:
# nhánh "coarse-grained" dùng CLIP tốt nhất khả thi, KHÔNG cần đúng tên [41]
# (paper cite CLIP gốc OpenAI nhưng không chỉ rõ checkpoint, nên chọn bản
# LAION mạnh nhất fit T4 là hợp lý — đã thảo luận và quyết định trước đó).
CLIP_MODEL       = "ViT-bigG-14"
CLIP_PRETRAIN    = "laion2b_s39b_b160k"
CLIP_BATCH       = 16
FAISS_BEIT3_DIM  = 1024  # verify thật: encoder_embed_dim trong _get_large_config
FAISS_CLIP_DIM   = 1280  # bigG-14 embed_dim, verify từ model config chính thức
NEIGHBOR_WINDOW  = 3

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

for d in [OUTPUT_ROOT, ZIP_DIR, INDEX_DIR, LOCAL_KF_DIR, SHARED_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f"[Config] VIDEO_ROOT  : {VIDEO_ROOT}")
print(f"[Config] OUTPUT_ROOT : {OUTPUT_ROOT}")
print(f"[Config] LOCAL buffer: {LOCAL_KF_DIR}")
print(f"[Config] ZIP every   : {ZIP_EVERY_GB} GB")

# ── Load / Init progress ───────────────────────────────────────────────────────
def load_progress() -> dict:
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f:
            p = json.load(f)
        print(f"[Resume] {len(p.get('done_videos',[]))} videos done, "
              f"{p.get('zip_count',0)} zips created.")
        return p
    print("[Resume] No progress file found – starting fresh.")
    return {
        "done_videos": [],
        "zip_count": 0,
        "completed_zips": [],
        "buffer_videos": [],
        "buffer_size_bytes": 0,
        "total_frames_cut": 0,
        "index_done_beit3": [],
        "index_done_clip":   [],
        "index_phase_done":  False,
        # Feature extraction phase (Part 2)
        "done_ocr":     [],
        "done_asr":     [],
        "done_caption": [],
        "done_tags":    [],
        "done_objects": [],
        "features_merged": False,
        "last_updated": None,
    }

def save_progress(p: dict):
    p["last_updated"] = datetime.now().isoformat()
    with open(PROGRESS_FILE, "w") as f:
        json.dump(p, f, indent=2, ensure_ascii=False)

PROGRESS = load_progress()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 02b – Buffer Recovery (quan trọng: phát hiện videos bị mất khi disconnect)
# ═══════════════════════════════════════════════════════════════════════════════
# %%
# Khi Colab disconnect, /content/kf_buffer bị xóa.
# Videos đã mark done nhưng chưa kịp zip → keyframes mất.
# Section này phát hiện và đưa chúng về hàng đợi lại.
# %%

def _find_unzipped_videos() -> set[str]:
    """Tìm video names đã có trong các completed_zips."""
    found = set()
    for zname in PROGRESS.get("completed_zips", []):
        zpath = os.path.join(ZIP_DIR, zname)
        if not os.path.exists(zpath): continue
        with zipfile.ZipFile(zpath) as zf:
            for name in zf.namelist():
                parts = name.split("/")
                if parts: found.add(parts[0])
    return found

_zipped_videos = _find_unzipped_videos()
_buffer_lost   = [v for v in PROGRESS.get("buffer_videos", [])
                  if v not in _zipped_videos]

if _buffer_lost:
    print(f"[Recovery] Found {len(_buffer_lost)} buffered videos lost on disconnect:")
    for v in _buffer_lost:
        print(f"  ↩ {v}")
    # Đưa về done_videos để cut lại
    for v in _buffer_lost:
        if v in PROGRESS["done_videos"]:
            PROGRESS["done_videos"].remove(v)
    PROGRESS["buffer_videos"]    = [v for v in PROGRESS["buffer_videos"]
                                    if v not in _buffer_lost]
    PROGRESS["buffer_size_bytes"] = 0
    save_progress(PROGRESS)
    print("[Recovery] Re-queued. They will be re-cut in Section 07.")
else:
    print("[Recovery] Buffer OK - no videos lost.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 03 – List Videos (filter theo MODE)
# ═══════════════════════════════════════════════════════════════════════════════
# %%

def list_all_videos(root: str) -> list[str]:
    vids = sorted(set(
        glob.glob(os.path.join(root, "**", "*.mp4"), recursive=True) +
        glob.glob(os.path.join(root, "*.mp4"))
    ))
    if not vids:
        raise FileNotFoundError(f"No videos found in {root}")
    return vids

_all_raw      = list_all_videos(VIDEO_ROOT)
# Filter theo MODE
# Cắt TOÀN BỘ video trong AIC_Video (L21–L30 và K01–K20), không lọc theo
# TEST_VIDEOS — việc chọn tập con là chuyện của bước đánh giá, không phải bước cắt.
ALL_VIDEOS = _all_raw

# ─── Lấy đúng 6 batch của phần này ──────────────────────────────────────────
# Lọc theo TIỀN TỐ tên video. BATCHES đặt ở SECTION 02a, 5 phần không giao nhau
# và gộp lại phủ đủ 30 batch.
_all_sorted    = sorted(ALL_VIDEOS)
_prefixes       = set(BATCHES)
ALL_VIDEOS = [v for v in _all_sorted if Path(v).stem.split("_")[0] in _prefixes]

from collections import Counter as _Cnt
_by_batch = _Cnt(Path(v).stem.split("_")[0] for v in ALL_VIDEOS)

print(f"\n[Part {PART}/5] got {len(ALL_VIDEOS)}/{len(_all_sorted)} videos")
print("[Batch] " + " · ".join(f"{n}:{_by_batch.get(n, 0)}" for n in BATCHES))
print("[First]    " + " ".join(Path(v).stem for v in ALL_VIDEOS[:8]))
print("[Last]     " + " ".join(Path(v).stem for v in ALL_VIDEOS[-8:]))

# Đối chiếu với số kỳ vọng lấy từ bảng fps của BTC — lệch nghĩa là Drive thiếu video
_empty = [n for n in BATCHES if _by_batch.get(n, 0) == 0]
if _empty:
    print(f"\n[!] Batches with NO videos found: {_empty}")
    print( "    Check VIDEO_ROOT, or Drive may still be syncing.")
if len(ALL_VIDEOS) != EXPECTED_VIDEOS:
    _missing = EXPECTED_VIDEOS - len(ALL_VIDEOS)
    print(f"\n[!] Expected {EXPECTED_VIDEOS} videos, found {len(ALL_VIDEOS)} "
          f"({'missing' if _missing > 0 else 'extra'} {abs(_missing)}).")
    print( "    Still runnable, but worth knowing before you assume it finished.")
else:
    print(f"\n[OK] All {EXPECTED_VIDEOS} videos present, matching the organiser fps table.")

done_set  = set(PROGRESS["done_videos"])
todo_list = [v for v in ALL_VIDEOS if Path(v).stem not in done_set]

total_gb = sum(os.path.getsize(v) for v in ALL_VIDEOS) / 1024**3
done_gb  = sum(os.path.getsize(v) for v in ALL_VIDEOS
               if Path(v).stem in done_set) / 1024**3

print(f"\n[Videos] MODE={MODE} | Total: {len(ALL_VIDEOS)} | Done: {len(done_set)} | Remaining: {len(todo_list)}")
print(f"[Size]   Total: {total_gb:.1f} GB | Done: {done_gb:.1f} GB")
print(f"[Sample] First 5 remaining:")
for v in todo_list[:5]:
    print(f"  {Path(v).stem}  ({os.path.getsize(v)/1024**2:.0f} MB)")

if not todo_list:
    print("\n[OK] All videos cut. Go to Section 09 to build the index.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 04 – Load TransNetV2 (auto-download weights nếu thiếu)
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import torch
from transnetv2_pytorch import TransNetV2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[System] Device: {DEVICE}")

if not os.path.exists(CKPT_TNV2):
    print(f"[TransNetV2] Weights not found → downloading from GDrive...")
    import gdown
    gdown.download(id=TNV2_GDRIVE_ID, output=CKPT_TNV2, quiet=False)
    print(f"[TransNetV2] Downloaded → {CKPT_TNV2}")

def load_tnv2(ckpt_path: str):
    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    m = TransNetV2()
    m.load_state_dict(state)
    return m.eval().to(DEVICE)

if todo_list:
    tnv2 = load_tnv2(CKPT_TNV2)
    print("[TransNetV2] Loaded OK")
else:
    print("[TransNetV2] Skipped (no videos remaining)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 06 – Helper Functions (giữ nguyên từ nb_01)
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import ffmpeg, cv2
from PIL import Image
from tqdm import tqdm

# ──── Video Decode 48×27 cho TransNetV2 ────────────────────────────────────────
def decode_video_tnv2(path: str) -> np.ndarray:
    stream, _ = (
        ffmpeg.input(path)
        .output("pipe:", format="rawvideo", pix_fmt="rgb24", s=f"{TNV2_W}x{TNV2_H}")
        .run(capture_stdout=True, capture_stderr=True)
    )
    return np.frombuffer(stream, np.uint8).reshape([-1, TNV2_H, TNV2_W, 3])

def get_video_fps(path: str) -> float:
    try:
        import json as _json
        r = _sp.run(["ffprobe", "-v", "error", "-select_streams", "v:0",
                     "-show_entries", "stream=r_frame_rate", "-of", "json", path],
                    capture_output=True, text=True, timeout=15)
        info = _json.loads(r.stdout)
        fr = info["streams"][0]["r_frame_rate"]
        n, d = fr.split("/"); return float(n) / float(d)
    except Exception:
        cap = cv2.VideoCapture(path); fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
        cap.release(); return fps

# ──── TransNetV2 Chunked (fix OOM cho video dài) ────────────────────────────────
def run_tnv2_chunked(frames: np.ndarray, model,
                     chunk: int = TNV2_CHUNK, ovl: int = TNV2_OVERLAP) -> np.ndarray:
    n = frames.shape[0]
    preds  = np.zeros(n, dtype=np.float32)
    counts = np.zeros(n, dtype=np.int32)
    start  = 0
    while start < n:
        end = min(start + chunk, n)
        t   = torch.from_numpy(frames[start:end].copy()).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            sp, _ = model(t)
            p = torch.sigmoid(sp).cpu().numpy().squeeze()
        p = np.atleast_1d(p).astype(np.float32)
        preds[start:end]  += p[:end-start]
        counts[start:end] += 1
        del t, sp, _
        torch.cuda.empty_cache()
        if end >= n: break
        start = end - ovl
    mask = counts > 0
    preds[mask] /= counts[mask]
    return preds

# ──── Scene Boundary ─────────────────────────────────────────────────────────────
def pick_cuts(prob: np.ndarray, thr=CUT_THR, min_d=MIN_CUT_DIST) -> list[int]:
    p, N = np.asarray(prob).ravel(), len(prob)
    mask, cuts = p >= thr, []
    i = 0
    while i < N:
        if mask[i]:
            j = i
            while j + 1 < N and mask[j+1]: j += 1
            cuts.append(i + int(np.argmax(p[i:j+1])))
            i = j + 1
        else:
            i += 1
    pruned = []
    for c in cuts:
        if not pruned or c - pruned[-1] >= min_d: pruned.append(c)
        elif p[c] > p[pruned[-1]]:               pruned[-1] = c
    return pruned

def cuts_to_scenes(cuts: list[int], n: int) -> list[tuple[int,int]]:
    cuts   = sorted(int(c) for c in cuts if 0 <= c < n)
    starts = [0]    + [c+1 for c in cuts]
    ends   = cuts   + [n-1]
    return [(s,e) for s,e in zip(starts,ends) if e-s+1 >= MIN_SCENE_LEN]

def sample_scene(s: int, e: int, n: int = N_SAMPLE_PER_SCENE) -> list[int]:
    if e - s + 1 <= n: return list(range(s, e+1))
    return np.linspace(s, e, n, dtype=int).tolist()

# ──── Alg.1 Frame Filtering (Paper §3.2.3, cos > 0.9 → near-duplicate) ──────────
@torch.no_grad()
def alg1_dedup_filter(frames_48x27: np.ndarray,
                      selected_per_scene: list[list[int]],
                      model, preprocess,
                      S: float = DEDUP_THR) -> list[list[int]]:
    all_idx = list({i for scene in selected_per_scene for i in scene})
    if not all_idx: return [[] for _ in selected_per_scene]
    emb_map: dict[int, np.ndarray] = {}
    for start in range(0, len(all_idx), 64):
        chunk_idx = all_idx[start:start+64]
        pils = [Image.fromarray(frames_48x27[i]) for i in chunk_idx]
        t    = torch.stack([preprocess(p) for p in pils]).to(DEVICE)
        f    = model.encode_image(t)
        f    = (f / f.norm(dim=-1, keepdim=True)).cpu().numpy()
        for ci, feat in zip(chunk_idx, f):
            emb_map[ci] = feat
    def cos(a, b): return float(np.dot(a, b))
    kept_all = []
    for scene in selected_per_scene:
        if not scene: kept_all.append([]); continue
        kept = [scene[0]]
        for t in range(1, len(scene)):
            e_t  = emb_map[scene[t]]
            drop = any(cos(e_t, emb_map[k]) >= S for k in kept)
            if not drop: kept.append(scene[t])
        kept_all.append(kept)
    return kept_all

# ──── Save Keyframes (cv2 seek → ffmpeg fallback cho H.265/HEVC) ──────────────────
def save_keyframes(video_path: str, kept_per_scene: list[list[int]],
                   video_name: str, fps: float, out_dir: str,
                   quality: int = JPEG_QUALITY) -> list[dict]:
    import subprocess as _sp2
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    target_map: dict[int, tuple[int,int]] = {}
    for sid, scene in enumerate(kept_per_scene):
        for rank, fi in enumerate(scene):
            target_map[fi] = (sid, rank)
    if not target_map: return []
    sorted_targets = sorted(target_map.keys())

    # Method 1: cv2 seek (fast, H.264 CFR)
    local_tmp = f"/tmp/_aic_tmp_{video_name}.mp4"
    if "/content/drive" in video_path and not os.path.exists(local_tmp):
        shutil.copy2(video_path, local_tmp)
    read_path = local_tmp if os.path.exists(local_tmp) else video_path
    frame_buf: dict[int, np.ndarray] = {}
    cap = cv2.VideoCapture(read_path)
    if cap.isOpened():
        n_cv2 = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        for fi in sorted_targets:
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
            ret, frm = cap.read()
            if ret: frame_buf[fi] = cv2.cvtColor(frm, cv2.COLOR_BGR2RGB)
        cap.release()
        if not frame_buf:
            print(f"  [WARN] cv2 seek 0/{len(sorted_targets)} frames "
                  f"(cv2_total={n_cv2}, max_target={sorted_targets[-1]}) "
                  f"→ H.265/HEVC detected, switching to ffmpeg select fallback")

    # Method 2: ffmpeg select (reliable, works H.265/HEVC)
    if not frame_buf:
        ffmp_tmp = f"/tmp/_aic_ffout_{video_name}"
        Path(ffmp_tmp).mkdir(exist_ok=True)
        try:
            sel = "+".join(f"eq(n\\,{fi})" for fi in sorted_targets)
            cmd = ["ffmpeg", "-y", "-hwaccel", "auto", "-i", read_path,
                   "-vf", f"select='{sel}'", "-vsync", "vfr", "-q:v", "2",
                   os.path.join(ffmp_tmp, "f_%06d.jpg")]
            _sp2.run(cmd, capture_output=True, timeout=900)
            extracted = sorted(glob.glob(os.path.join(ffmp_tmp, "f_*.jpg")))
            for fi, fpath in zip(sorted_targets, extracted):
                if os.path.exists(fpath) and os.path.getsize(fpath) > 0:
                    frame_buf[fi] = np.array(Image.open(fpath).convert("RGB"))
            print(f"  [FFmpeg] Extracted {len(frame_buf)}/{len(sorted_targets)} frames")
        except Exception as e:
            print(f"  [ERROR] ffmpeg fallback failed: {e}")
        finally:
            shutil.rmtree(ffmp_tmp, ignore_errors=True)

    if os.path.exists(local_tmp): os.remove(local_tmp)
    if not frame_buf:
        print(f"  [ERROR] Both methods failed for {video_name}"); return []

    first = next(iter(frame_buf.values()))
    H_img, W_img = first.shape[:2]
    meta_list = []
    for fi, (sid, rank) in sorted(target_map.items(), key=lambda x: (x[1][0], x[1][1])):
        if fi not in frame_buf: continue
        fname    = f"{video_name}-{sid:04d}-{fi}.jpg"
        ts_ms    = int(fi / fps * 1000)
        ts_str   = (f"{ts_ms//3600000:02d}:{(ts_ms%3600000)//60000:02d}:"
                    f"{(ts_ms%60000)//1000:02d}.{ts_ms%1000:03d}")
        fpath    = os.path.join(out_dir, fname)
        Image.fromarray(frame_buf[fi]).save(fpath, "JPEG", quality=quality)
        meta_list.append({
            "frame":         f"/static/images/{fname}",
            "url":           f"http://localhost:8000/static/images/{fname}",
            "name":          fname,
            "video":         video_name,
            "scene_id":      sid,
            "frame_idx":     fi,
            "fps":           fps,
            "width":         W_img,
            "height":        H_img,
            "timestamp_ms":  ts_ms,
            "timestamp_str": ts_str,
            "content": [], "ocr": [], "object": [], "color": [], "action": [],
            "faiss_id_clip":   -1,
            "faiss_id_beit3":  -1,
            "neighbors_clip":  [],
        })
    return meta_list

# ──── ZIP & Upload to Drive ───────────────────────────────────────────────────────
def zip_and_upload(video_names: list[str], zip_id: int,
                   local_kf_dir: str, zip_dir: str) -> str:
    zip_name = f"AIC2026_frames_p{PART}_{zip_id:03d}.zip"   # tách theo phần
    zip_path = os.path.join(zip_dir, zip_name)
    print(f"  [Zip] Creating {zip_name} ({len(video_names)} videos)...")
    t0 = time.time()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED) as zf:
        for vn in video_names:
            vdir = os.path.join(local_kf_dir, vn)
            if not os.path.isdir(vdir): continue
            for f in os.listdir(vdir):
                zf.write(os.path.join(vdir, f), arcname=f"{vn}/{f}")
    sz_mb = os.path.getsize(zip_path) / 1024**2
    print(f"  [Zip] {zip_name}: {sz_mb:.0f} MB in {time.time()-t0:.0f}s")
    for vn in video_names:
        vdir = os.path.join(local_kf_dir, vn)
        if os.path.isdir(vdir): shutil.rmtree(vdir)
    return zip_name

print("[SECTION 06] Helper functions OK")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 06b – Lấy mẫu theo tốc độ  (ghi đè hàm của SECTION 06)
# ═══════════════════════════════════════════════════════════════════════════════
# %%
# Số ảnh phụ thuộc ĐỘ DÀI cảnh, không cố định 4. Cảnh 8 frame và cảnh 4500 frame
# ở cách cũ đều được đúng 4 ảnh — cảnh dài bị bỏ sót nội dung.
# Và KHÔNG lọc trùng ở bước cắt: paper §3.2.3 lọc SAU khi trích đặc trưng, dùng
# chính embedding nạp FAISS. Lọc trên buffer 48×27 thì chữ trên màn hình đã biến
# mất trước khi so, nên nó xoá ngẫu nhiên.

FLOOR_SECONDS   = 50 / 30    # ≈ 1,67 giây — hết sàn 2 ảnh
SECONDS_PER_EXTRA = 2.0        # sau sàn, cứ 2 giây thêm 1 ảnh
N_MIN, N_MAX   = 2, 40
MARGIN         = 3          # lùi 3 frame mỗi đầu, tránh frame chuyển cảnh

from collections import Counter
import open_clip   # SECTION 05 (lọc trùng) đã bỏ, mà dòng import nằm trong đó

# ─── Bảng fps của BTC ────────────────────────────────────────────────────────
# BTC dùng CHÍNH bảng này để đánh số frame trong đáp án. ffprobe có thể trả 30.0
# ở video mà BTC ghi 29.97 — lệch 20%, tới phút 20 là sai gần 4 phút, đủ trượt
# hẳn cảnh chứa đáp án. Nên bảng luôn được ưu tiên.
FPS_MAP: dict[str, float] = {}
FPS_MAP_PATH = f"{SHARED_DIR}/fps_map.json"
if os.path.exists(FPS_MAP_PATH):
    with open(FPS_MAP_PATH) as f:
        FPS_MAP = {k: float(v) for k, v in json.load(f).items()}
    print(f"[fps] Organiser table: {len(FPS_MAP)} videos · " +
          " · ".join(f"{k}fps:{v}" for k, v in sorted(Counter(FPS_MAP.values()).items())))
else:
    print(f"[fps] MISSING {FPS_MAP_PATH}")
    print( "      Copy data_raw/fps_map.json from the repo there - otherwise EVERY video")
    print( "      must be probed with ffprobe and frame numbers may drift from the answers.")

# Bảng đã phủ ĐỦ 1478 video (873 của L21–L30 + 605 của K01–K20), đã đối chiếu với
# ffprobe trên từng file: 1478/1478 khớp, lệch lớn nhất 0,002 fps. Nhánh dò dưới đây
# là lưới an toàn cho video lạ, bình thường sẽ không bao giờ chạy tới.
#
# LƯU Ý: đừng "sửa" 29.97 thành 30000/1001 cho chính xác hơn — dùng 29.97 mới tái lập
# đúng frame_idx của BTC (100%), còn giá trị chính xác lệch 1 frame ở 54% số dòng.
_FPS_PROBED = {}

def get_video_fps(path: str) -> float:
    """Ghi đè hàm SECTION 06. Ưu tiên bảng BTC, thiếu thì mới dò."""
    ten = Path(path).stem
    if ten in FPS_MAP:
        return FPS_MAP[ten]
    try:
        r = _sp.run(["ffprobe", "-v", "error", "-select_streams", "v:0",
                     "-show_entries", "stream=r_frame_rate", "-of", "json", path],
                    capture_output=True, text=True, timeout=15)
        n, d = json.loads(r.stdout)["streams"][0]["r_frame_rate"].split("/")
        fps = float(n) / float(d)
    except Exception:
        cap = cv2.VideoCapture(path)
        fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
        cap.release()
    _FPS_PROBED[ten] = fps
    return fps


def sample_scene(s: int, e: int, fps: float = 25.0) -> list[int]:
    """Số ảnh theo độ dài cảnh.

    n = 2 cho tới 1,67 giây; sau đó cứ thêm 2 giây thì thêm 1 ảnh; kẹp [2, 40].
    Quy ra frame @30fps:  8–50 → 2 ảnh · 51–110 → 3 · 111–170 → 4
                 @25fps:  8–42 → 2 ảnh · 43–92  → 3 · 93–142  → 4
    Mốc theo GIÂY giống nhau ở mọi fps, nên video 25 và 30 fps có cùng mật độ ảnh
    trên mỗi giây.
    """
    L = e - s + 1
    _floor_frames  = max(1, int(round(fps * FLOOR_SECONDS)))
    _step_frames = max(1, int(round(fps * SECONDS_PER_EXTRA)))
    n = N_MIN if L <= _floor_frames else N_MIN + -(-(L - _floor_frames) // _step_frames)
    n = max(N_MIN, min(N_MAX, n))
    if L <= n:
        return list(range(s, e + 1))
    m = min(MARGIN, (L - n) // 2)
    return np.linspace(s + m, e - m, n, dtype=int).tolist()


_in_table = sum(1 for v in todo_list if Path(v).stem in FPS_MAP)
_missing   = len(todo_list) - _in_table
print(f"\n[fps] {_in_table}/{len(todo_list)} videos found in the organiser table")
if _missing:
    print(f"[fps] WARN {_missing} videos NOT in the table - will be probed with ffprobe.")
    print( "      The table should cover all 1478 videos. Missing means the copy on Drive is stale,")
    print( "      re-copy the latest data_raw/fps_map.json from the repo before continuing.")
else:
    print( "[fps] OK every video is in the table - nothing needs probing.")
print(f"[Sampling] n=2 up to {FLOOR_SECONDS:.2f}s, then +1 frame every {SECONDS_PER_EXTRA}s, "
      f"clamp [{N_MIN}, {N_MAX}], inset {MARGIN} frames")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 07 – Vòng lặp cắt (auto-resume + auto-zip)
# ═══════════════════════════════════════════════════════════════════════════════
# %%

if not todo_list:
    print("[SKIP] All videos already cut. Go to Section 09.")
else:
    BUFFER_META: list[dict] = []
    # FPS_MAP là bảng BTC nạp ở SECTION 06b — KHÔNG đè.
    # FPS_USED ghi lại fps thực dùng cho từng video, để bước index dùng lại.
    FPS_USED:     dict[str, float] = {}
    fps_map_path = os.path.join(INDEX_DIR, "fps_map.json")

    print(f"\n[Main Loop] {len(todo_list)} videos | Buffer trigger: {ZIP_EVERY_GB} GB\n")

    for video_path in tqdm(todo_list, desc="Cutting"):
        video_name = Path(video_path).stem
        out_dir    = os.path.join(LOCAL_KF_DIR, video_name)
        t0         = time.time()
        print(f"\n  [{video_name}] START")

        fps = get_video_fps(video_path)
        FPS_USED[video_name] = fps

        try:
            frames_tnv2 = decode_video_tnv2(video_path)
        except Exception as e:
            print(f"  [ERROR] decode {video_name}: {e}"); continue

        n_frames = frames_tnv2.shape[0]
        print(f"  [1/4] Decoded 48×27: {n_frames} frames ({n_frames/fps:.0f}s)")

        prob   = run_tnv2_chunked(frames_tnv2, tnv2)
        cuts   = pick_cuts(prob)
        scenes = cuts_to_scenes(cuts, n_frames)
        print(f"  [2/4] TransNetV2: {len(cuts)} cuts → {len(scenes)} scenes")

        # Số ảnh theo độ dài cảnh, KHÔNG lọc trùng ở đây.
        sampled = [sample_scene(s, e, fps) for s, e in scenes]
        kept    = sampled
        _total   = sum(len(x) for x in kept)
        _dist   = Counter(len(x) for x in kept)
        print(f"  [3/4] Rate-based sampling: {_total} frames · "
              + " · ".join(f"{k}f x{v}" for k, v in sorted(_dist.items())[:6]))
        del frames_tnv2; torch.cuda.empty_cache()

        meta = save_keyframes(video_path, kept, video_name, fps, out_dir)
        print(f"  [4/4] Saved {len(meta)} keyframes → {out_dir}")

        PROGRESS["done_videos"].append(video_name)
        PROGRESS["buffer_videos"].append(video_name)
        PROGRESS["total_frames_cut"] += len(meta)
        buf_size = sum(
            os.path.getsize(os.path.join(out_dir, f))
            for f in os.listdir(out_dir) if os.path.isfile(os.path.join(out_dir, f))
        ) if os.path.isdir(out_dir) else 0
        PROGRESS["buffer_size_bytes"] += buf_size
        BUFFER_META.extend(meta)
        save_progress(PROGRESS)
        with open(fps_map_path, "w") as f: json.dump(FPS_USED, f, indent=2)
        print(f"  [{video_name}] DONE in {time.time()-t0:.0f}s | "
              f"Buffer: {PROGRESS['buffer_size_bytes']/1024**3:.2f} GB")

        if PROGRESS["buffer_size_bytes"] >= ZIP_EVERY_GB * 1024**3:
            zid      = PROGRESS["zip_count"] + 1
            zip_name = zip_and_upload(PROGRESS["buffer_videos"], zid, LOCAL_KF_DIR, ZIP_DIR)
            PROGRESS["zip_count"]         = zid
            PROGRESS["completed_zips"].append(zip_name)
            PROGRESS["buffer_videos"]     = []
            PROGRESS["buffer_size_bytes"] = 0
            BUFFER_META = []
            save_progress(PROGRESS)
            print(f"  [Zip] ✓ {zip_name}")

    if PROGRESS["buffer_videos"]:
        zid      = PROGRESS["zip_count"] + 1
        zip_name = zip_and_upload(PROGRESS["buffer_videos"], zid, LOCAL_KF_DIR, ZIP_DIR)
        PROGRESS["zip_count"]         = zid
        PROGRESS["completed_zips"].append(zip_name)
        PROGRESS["buffer_videos"]     = []
        PROGRESS["buffer_size_bytes"] = 0
        save_progress(PROGRESS)
        print(f"[Final Zip] ✓ {zip_name}")

    print(f"\n[✓] Cutting done! Total: {PROGRESS['total_frames_cut']:,} keyframes, "
          f"{PROGRESS['zip_count']} zips")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 08 – Status Check (run bất kỳ lúc nào)
# ═══════════════════════════════════════════════════════════════════════════════
# %%

p = load_progress()
zip_gb = sum(os.path.getsize(os.path.join(ZIP_DIR, z)) / 1024**3
             for z in p["completed_zips"]
             if os.path.exists(os.path.join(ZIP_DIR, z)))
print("=" * 60)
print(f"  PIPELINE STATUS  [{MODE} mode]")
print("=" * 60)
print(f"  Videos done        : {len(p['done_videos'])} / {len(ALL_VIDEOS)}")
print(f"  Total keyframes    : {p['total_frames_cut']:,}")
print(f"  Zips on Drive      : {p['zip_count']} ({zip_gb:.1f} GB)")
print(f"  BEiT3 indexed      : {len(p.get('index_done_beit3', []))} / {p['zip_count']}")
print(f"  CLIP indexed       : {len(p['index_done_clip'])} / {p['zip_count']}")
print(f"  Index done         : {p['index_phase_done']}")
print(f"  OCR done           : {len(p.get('done_ocr', []))} videos")
print(f"  ASR done           : {len(p.get('done_asr', []))} videos")
print(f"  Caption done       : {len(p.get('done_caption', []))} videos")
print(f"  Features merged    : {p.get('features_merged', False)}")
print("=" * 60)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 09 – Tổng kết phần này
# ═══════════════════════════════════════════════════════════════════════════════
# %%

p = load_progress()
print("=" * 72)
print(f"  PART {PART} · {' '.join(BATCHES)} · {VERSION}")
print("=" * 72)
print(f"  Videos cut        : {len(p['done_videos'])}/{len(ALL_VIDEOS)}")
print(f"  Remaining         : {len(ALL_VIDEOS) - len(p['done_videos'])}")
print(f"  Total frames      : {p.get('total_frames_cut', 0):,}")
print(f"  Zip files         : {len(p.get('completed_zips', []))}")
print(f"  Pending zip       : {len(p.get('buffer_videos', []))} videos · "
      f"{p.get('buffer_size_bytes', 0)/1024**3:.2f} GB")
print(f"  Output dir        : {OUTPUT_ROOT}")

if _FPS_PROBED:
    print()
    print(f"  [!] {len(_FPS_PROBED)} videos NOT in the fps table, probed with ffprobe:")
    for _t, _f in sorted(_FPS_PROBED.items())[:10]:
        print(f"        {_t}: {_f:.3f} fps")
    if len(_FPS_PROBED) > 10:
        print(f"        ... and {len(_FPS_PROBED) - 10} more")
    print( "      The table should cover all 1478 videos, so the copy on Drive is stale.")
    print( "      Re-copy the latest data_raw/fps_map.json and RE-CUT these videos -")
    print( "      their frame numbers may drift from the official answers.")
else:
    print()
    print("  OK every video used fps from the organiser table, none probed.")

if len(p["done_videos"]) >= len(ALL_VIDEOS):
    print()
    print("  DONE with this part. Tell the team lead to merge the zips and build the index.")
print("=" * 72)